In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from yellowbrick.classifier import ConfusionMatrix
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, KFold
from sklearn import tree
from statsmodels.stats.multicomp import MultiComparison
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', na_filter = False, header= None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', na_filter = False, header= None)
y_train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', na_filter = False, header= None)

In [ ]:
#Rename the Columns
for i in train:
 train.rename({i: f'Col_{i}'}, axis=1, inplace=True)

In [ ]:
#Searching Null Values (Summarizing the Total)
train.isnull().sum().sum()

In [ ]:
#Searching Empty Values
train_empty = []
train_col = train.columns
for col in train_col:
 x = train.loc[train[col] == ' '].count().sum()
 train_empty.append(x)
print(sum(train_empty))

#Showing the results in a DataFrame
df_empty_values = pd.DataFrame(columns = ['Columns','Empty_Values'])
df_empty_values['Columns'] = train_col
df_empty_values['Empty_Values'] = train_empty
df_empty_values

In [ ]:
#Searching Duplicate Values
train.loc[train.duplicated()].sum().sum()

In [ ]:
#Searching Outliers (Plot)
for i in train_col:
 print(i)
 plot = px.box(train, y = i)
 plot.show()

In [ ]:
#Verifying the Normal Distribution (Training Base)
for i in train_col:
  train[i].plot(kind='kde')

PCA

In [ ]:
X_train = train.iloc[:, 0:40].values
X_teste = test.iloc[:, 0:40].values
list_var_sum = []

#Test Differents n_componentes in PCA
for i,j in enumerate(train):
 pca = PCA(n_components=i)
 X_train_treinamento_pca_temp = pca.fit_transform(X_train)
 X_teste_testes_pca_temp = pca.fit_transform(X_teste)
 X_train_treinamento_pca_temp.shape, X_teste_testes_pca_temp.shape
 pca.explained_variance_ratio_
 var = pca.explained_variance_ratio_.sum()
 list_var_sum.append(var)

In [ ]:
#Showing the results in a DataFrame
df_pca_var = pd.DataFrame(columns = ['Var'])
df_pca_var['Var'] = list_var_sum
df_pca_var

In [ ]:
#Showing the results in a Plot
df_pca_var['Var'].plot(kind='line', title ='Variance x n_components (PCA) ')

In [ ]:
#Choosing a n_components (PCA) - Getting the index closest to 90% Variance
closest_to = min(list_var_sum, key=lambda x:abs(x-0.90))
n_comp_number = list_var_sum.index(closest_to)
pca = PCA(n_components=n_comp_number)
X_train_treinamento_pca_perm = pca.fit_transform(X_train)
X_teste_testes_pca_perm = pca.transform(X_teste)
X_train_treinamento_pca_perm.shape, X_teste_testes_pca_perm.shape, pca.explained_variance_ratio_.sum()

In [ ]:
#Verifying the balanced (Training Labels)
## Counting the Unique Values
y_train.value_counts()

Cross Validation: RandomTreeFlorest + SVM + KNN

In [ ]:
resultados_random_forest = []
resultados_knn = []
resultados_svm = []
for i in range(5):
  print(i)
  kfold = KFold(n_splits=10, shuffle=True, random_state=i)

  random_forest = RandomForestClassifier(criterion = 'entropy', min_samples_leaf = 1, min_samples_split=5, n_estimators = 10)
  random_forest.fit(X_train_treinamento_pca_perm, y_train)
  scores = cross_val_score(random_forest, X_train_treinamento_pca_perm, y_train, cv = kfold)
  resultados_random_forest.append(scores.mean())

  knn = KNeighborsClassifier(n_neighbors = 20, p = 1)
  knn.fit(X_train_treinamento_pca_perm, y_train)
  scores = cross_val_score(knn, X_train_treinamento_pca_perm, y_train, cv = kfold)
  resultados_knn.append(scores.mean())

  svm = SVC(kernel = 'rbf', C = 2.0)
  svm.fit(X_train_treinamento_pca_perm, y_train)
  scores = cross_val_score(svm, X_train_treinamento_pca_perm, y_train, cv = kfold)
  resultados_svm.append(scores.mean())

resultados = pd.DataFrame({'Random forest': resultados_random_forest, 'KNN': resultados_knn, 'SVM': resultados_svm})
resultados
resultados.describe()
resultados.var()
(resultados.std() / resultados.mean()) * 100

Statistical tests

In [ ]:
#Shapiro Wilkins Normality Test
##H0 = The data is normal distributed, if p-value < 0,05, we reject H0
from scipy.stats import shapiro
alpha = 0.05
shapiro(resultados_random_forest), shapiro(resultados_knn), shapiro(resultados_svm)

In [ ]:
#The plot
sns.displot(resultados_random_forest, kind = 'kde')
sns.displot(resultados_knn, kind = 'kde')
sns.displot(resultados_svm, kind = 'kde')

In [ ]:
#ANOVA
##H0 -> The algoritm results are the statistically equal
from scipy.stats import f_oneway
_, p = f_oneway(resultados_random_forest, resultados_knn, resultados_svm)
p
if p <= alpha:
  print('Null hypothesis rejected: data is different')
else:
  print('Alternative hypothesis rejected: data is equal')

In [ ]:
# Tukey

## H0 -> The results of the algoritm are the equal
### Rejection -> there is a significal statisticall diference
#### Not Rejection -> there is not a significal statisticall diference between the results of the algoritms

resultados_algoritmos = {'accuracy': np.concatenate([resultados_random_forest, resultados_knn, resultados_svm]),
                    'algoritm':['random_forest','random_forest','random_forest','random_forest','random_forest',
                                'knn','knn','knn','knn','knn',
                                'svm','svm','svm','svm','svm']}

resultados_df = pd.DataFrame(resultados_algoritmos)
resultados_df

compara_algoritmos = MultiComparison(resultados_df['accuracy'], resultados_df['algoritm'])
teste_estatistico = compara_algoritmos.tukeyhsd()
print(teste_estatistico)
resultados.mean()
teste_estatistico.plot_simultaneous()

Combination of Classifiers

In [ ]:
previsoes_random_forest1 = random_forest.predict(X_teste_testes_pca_perm)
previsoes_svm1 = svm.predict(X_teste_testes_pca_perm)
previsoes_knn1 = knn.predict(X_teste_testes_pca_perm)
forecast =[]
for i in previsoes_random_forest1:
  prev_1 = 0
  prev_0 = 0
  if previsoes_random_forest1 [i] == 1:
    prev_1 += 1
  else:
    prev_0 += 1
  if previsoes_svm1[i] == 1:
    prev_1 += 1
  else:
    prev_0 += 1
  if previsoes_knn1[i] == 1:
    prev_1 += 1
  else:
    prev_0 += 1
  if prev_1 > prev_0:
    forecast.append(1)
  else:
    forecast.append(0)

Submission

In [ ]:
submission = pd.DataFrame(columns = ['Solution'])
submission['Solution'] = forecast
submission.insert(0, 'Id', range(1, 1 + len(submission)))
submission.to_csv('submission.csv',index=False)